# دفتر تنظيف وتدقيق جودة بيانات الوظائف التقنية — التشغيل الكامل

- **تاريخ التدقيق:** 2026-10-28
- **مرجع المصادر:** `main@6755883a8c043592d349cace4cd84aaae81d73d0+GoogleDrive:International_Tech_Jobs_Filtered.zip`
- **الصفوف قبل/بعد:** 775,155 / 775,155
- **الصفوف المحذوفة:** 0

هذا الدفتر يوثق التنظيف على مصادر التشغيل المسجل. ملفات المصدر للقراءة فقط؛ كل صف خام محفوظ في `source_row_json`. لا تُدمج الإعلانات أو المهارات دلاليًا ولا تُحذف outliers. ملف الوظائف المنظف يبقى في Drive ولا يُضم إلى Git؛ عند استعادته محليًا إلى `data/processed/` يستطيع الدفتر فحص عدد صفوفه تدفقيًا.


## 1. النطاق ومصدر البيانات

تُسجل المصادر المختارة ونسخ base المستخدمة لمقارنة إثراء المهارات في `source_manifest.csv`. لا تُضاف صفوف نسخ base مرة أخرى. أي اختلاف في timestamp خارجي عن تاريخ التدقيق يُذكر كقيد provenance، ولا يُعامل تاريخًا للنشر أو الجمع.

## 2. سياسة التغييرات

الأصل محفوظ؛ التنظيف والتوحيد في أعمدة مشتقة. التطبيع النصي لا يوحد معاني العناوين أو البلدان تقريبيًا. التكرارات exact/near كلها مرشحة للمراجعة فقط ولا تُحذف. إعلانات `updated` منفصلة عن `posted`، ولا تُستنتج تواريخ الجمع من timestamps خارجية.


## 3. الرواتب: التحويل والتقدير

- `salary_original` / `salary_raw` تحفظ القيمة المصدرية كما هي؛ أي قيمة مضافة مستقلة في أعمدة سنوية مشتقة.
- عوامل السنونة الصريحة: شهر ×12.0, أسبوع ×52.0, يوم عمل ×260.0, ساعة ×2080.0. العوامل افتراضات موثقة وليست حقائق من المصدر.
- القيمة الموحدة الأساسية USD؛ تُشتق EGP وSAR باستخدام سياسة الصرف المؤرخة. لا يوجد تعديل تضخم أو PPP.
- مرجع التقدير يستخدم فقط `actual_reported` مع `salary_statistics_eligible=yes`، وعملة وفترة صريحتين وقيمة سنوية USD موجبة صالحة؛ استُخدمت 3,354 سجلات ضمن 2,395 مجموعات.
- التقديرات المضافة في هذا التشغيل: 715,390. كل قيمة Synthetic موسومة Estimated مع المنهج والثقة وعدد المراجع؛ نطاق Q1–Q3 مرجع Synthetic داخلي وليس نطاق راتب معلن.
- `Not disclosed` والنص غير القابل للتحليل والعملة/الفترة الغامضة لا تُخمن لها قيم. القيم الشاذة محفوظة وموسومة فقط.


In [ ]:
from pathlib import Path
import csv
import gzip
import json

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'scripts' / 'data_cleaning' / 'finalize_full_cleaning_audit.py').is_file():
            return candidate
    raise FileNotFoundError('Repository root not found')

PROJECT_ROOT = find_project_root()
REPORT_DIR = PROJECT_ROOT / 'analysis' / 'reports' / 'data_cleaning'
DATASET = PROJECT_ROOT / 'data' / 'processed' / 'jobs_cleaned_audited.csv.gz'
SUMMARY = json.loads((REPORT_DIR / 'cleaning_summary.json').read_text(encoding='utf-8'))
BEFORE = json.loads((REPORT_DIR / 'data_quality_before.json').read_text(encoding='utf-8'))
AFTER = json.loads((REPORT_DIR / 'data_quality_after.json').read_text(encoding='utf-8'))
print('scope:', SUMMARY['scope'])
print('rows before / after / dropped:', SUMMARY['input_rows'], SUMMARY['output_rows'], SUMMARY['rows_dropped'])

if DATASET.is_file():
    print('dataset bytes:', DATASET.stat().st_size)
    streamed_rows = 0
    missing_core_fields = 0
    with gzip.open(DATASET, 'rt', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        required = {'record_id', 'source_row_json'}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f'Missing required audit columns: {sorted(required - set(reader.fieldnames or []))}')
        for row in reader:
            streamed_rows += 1
            if not row.get('record_id') or not row.get('source_row_json'):
                missing_core_fields += 1
    assert streamed_rows == int(SUMMARY['output_rows']), (streamed_rows, SUMMARY['output_rows'])
    assert missing_core_fields == 0, missing_core_fields
    print('stream-validated rows:', streamed_rows, '| missing core fields:', missing_core_fields)
else:
    print('Cleaned dataset is not in this Git checkout. Restore jobs_cleaned_audited.csv.gz from Drive to data/processed/ to run the streaming row-count check.')

## 4. التغطية والحقول والأنواع

`raw_source_column_coverage.csv` و`cleaned_column_coverage.csv` و`field_coverage_comparison.csv` تلخص المفقود حسب المصدر/الحقل. `cleaned_column_type_audit.csv` يفحص أعمدة الناتج؛ وتُعرض خلاصاتها هنا، دون تضمين الداتا المنظفة نفسها في Git.

In [ ]:
def read_csv(path):
    with Path(path).open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

missing = read_csv(REPORT_DIR / 'missingness_impact.csv')
print('Highest-impact fields with the most original missing values:')
for row in sorted(missing, key=lambda r: int(r['missing_original_count'] or 0), reverse=True)[:12]:
    print(row['source_file'], row['logical_field'], row['missing_original_count'], row['missing_original_percent'] + '%')
types = read_csv(REPORT_DIR / 'cleaned_column_type_audit.csv')
print('Output columns checked:', len(types), '| invalid typed values:', sum(int(r['invalid_type_values']) for r in types))

## 5. التواريخ والتكرارات والمهارات

يسجل التقرير النهائي حدود فحص تواريخ النشر والتحديث والجمع، والأعداد المجمعة للتكرارات والمرشحين. لا يثبت التشابه أن الإعلان مكرر، ولا تُحذف أو تُدمج أي صفوف. تُحفظ تفاصيل المراجعة الصفّية وجداول المهارات الكبيرة كمخرجات محلية مولّدة ولا تُضاف إلى Git.

In [ ]:
duplicate_review = SUMMARY.get('full_data_quality_audit', {}).get('duplicate_review', {})
for key in (
    'candidate_groups_total',
    'cross_source_exact_raw_row_groups',
    'cross_source_exact_url_groups',
    'cross_source_exact_listing_signature_groups',
    'cross_source_near_duplicate_pairs',
    'large_near_duplicate_blocks_skipped_over_60_rows',
    'deleted_or_merged_rows',
):
    print(key, duplicate_review.get(key, 'not_available'))

checks = read_csv(REPORT_DIR / 'field_consistency_checks.csv')
for row in checks:
    print(row['check_name'], row['status'], 'violations=', row['violations'])

## 6. الرواتب والتحليلات المُعادة

الراتب الأصلي يبقى محفوظًا، وتُفصل القيم المصدرية الفعلية عن القيم السنوية المحولة وعن التقديرات الاصطناعية الموسومة. يُراجع المنهج والحدود في `analysis/reports/data_cleaning/final_data_quality_report.md`؛ أسعار الصرف والسنونة المؤرخة في `data/reference/fx_rates_2026-10-28.json`. جداول الرواتب الصفّية والتفصيلية ليست ضمن Git.

## 7. التشغيل وإعادة الإنتاج

```bash
python scripts/data_cleaning/build_cleaning_audit.py   --raw-data-directory /path/to/restored/data/raw   --dataset-output data/processed/jobs_cleaned_audited.csv.gz   --report-directory analysis/reports/data_cleaning   --source-ref 'main@6755883a8c043592d349cace4cd84aaae81d73d0+GoogleDrive:International_Tech_Jobs_Filtered.zip'   --as-of-date 2026-10-28 --require-all-sources
python scripts/data_cleaning/finalize_full_cleaning_audit.py   --raw-data-directory /path/to/restored/data/raw   --dataset data/processed/jobs_cleaned_audited.csv.gz   --report-directory analysis/reports/data_cleaning   --source-ref 'main@6755883a8c043592d349cace4cd84aaae81d73d0+GoogleDrive:International_Tech_Jobs_Filtered.zip'   --as-of-date 2026-10-28   --fx-policy-file data/reference/fx_rates_2026-10-28.json
```

يجب أن يحتوي المجلد على كل المصادر المحددة ونسخ base اللازمة. لا تُوصف مجموعة جزئية بأنها كاملة. بيانات المصدر والمنظفة موجودة في Drive وليستا جزءًا من نشر ملفات التنظيف على GitHub.
